In this notebook, we load models trained with best hyperparameters and analyze their performance.

In [ ]:
import json
import keras
import torch

from load_dataset import build_dataset, make_tf_dataset, make_torch_dataset ########################################################################## Uncomment once metrics are implemented
from metric import *

# Load trained models

In [6]:
unet_model_path = "PATH HERE" # ex: modeL_out.keras
swin_model_path = "PATH HERE" # ex: 'model_weights.pth'

unet_model = keras.models.load_model(unet_model_path)
swin_model = torch.load(swin_model_path, weights_only=True)

ValueError: File format not supported: filepath=PATH HERE. Keras 3 only supports V3 `.keras` files, legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(PATH HERE, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).

# Load test dataset

In [4]:
x_test, y_test = build_dataset(data="test")

NameError: name 'build_dataset' is not defined

# Inference by U-Net

In [ ]:
y_test_pred = unet_model.predict(new_data_samples)

# calculate metrics

# Infernece by Swin-transformer

In [ ]:
model.eval()
with torch.inference_mode():
    # TODO: this may not be the right usages
    test_ds = make_torch_dataset(x_test, y_test, hparams["batch_size"], training=False)

    y_test_pred = swin_model(test_ds)

    # calculate metrics

# Plot to compare the results

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Helper functions to load training history
def load_history(model_name: str):
    """
    Load the training history for a model from either a saved JSON file or a variable
    in memory (e.g. previous training run)

    Parameters:
        model_name (str): Name of the model (unet or swin)
    
    Returns:
        dict: dictionary with keys like loss, val_loss, dice_coef, val_dice_coef,
            containing lists of values (one per training epoch)
    """

    # Try to find a saved history file
    history_files = [
        Path(f"{model_name}_history.json"),
        Path(f"{model_name.lower()}_history.json"),
        Path(f"{model_name.replace(' ', '_').lower()}_history.json"),
    ]

    for history_file in history_files:
        if history_file.exists():
            with history_file.open("r", encoding="utf-8") as handle:
                return json.load(handle)
    
    # If no file found, try to find a history object in memory from a prev run in this notebook
    history_vars = {"U-Net": "unet_history", "Swin Transformer": "swin_history"}
    history_obj = globals().get(history_vars.get(model_name, ""))
    if hasattr(history_obj, "history"):
        return history_obj.history
    if isinstance(history_obj, dict):
        return history_obj
    return None

def history_series(history: dict, key: str):
    """
    Extract a specific metric from the history dictionary.

    Example: If history contains training and validation loss:
    - history_series(history, "loss") returns [0.5, 0.4, 0.3, ...]
    - history_series(history, "val_loss") returns [0.6, 0.5, 0.4, ...]

    Parameters:
        history (dict): the training history dict
        key (str): the metric to extract (e.g. loss, val_dice_coef)
    
    Returns:
        list or None: values for that metric across epochs, or None if not found
    """
    if history is None:
        return None
    values = history.get(key)

    # If validation metric not found, try the training metric instead
    if values is None and key.startswith("val_"):
        values = history.get(key[4:])
    return values

# Loading training histories for both models

# Try to load saved history for U-Net and Swin Transformer
histories = {
    "U-Net": load_history("unet"),
    "Swin Transformer": load_history("swin"),
}

# Keep only models that have history data available, i.e. remove any that returned None
histories = {}
for name, history in histories.items():
    if history is not None:
        histories[name] = history


# Plot training curves (loss and dice coefficient)
if not histories:
    print("No saved training histories found. Save each model's history as unet_history.json and swin_history.json to plot learning curves.")
else:
    # Create a figure with 2 subplots side by side
    # Subplot 1: loss curves
    # Subplot 2: dice curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Define which metrics to plot
    # Each tuple is (training_metric, validation_metric, plot_title)
    curve_specs = [
        ("loss", "val_loss", "Loss"),
        ("dice_coef", "val_dice_coef", "Dice"),
    ]

    # Define colors
    colors ={"U-Net": "tab:blue", "Swin Transformer": "tab:orange"}

    # Loop through each metric (loss and dice)
    for ax, (train_key, val_key, title) in zip(axes, curve_specs):
        plotted = False # Track if we plotted anything

        # Loop through each model
        for model_name, history in histories.items():
            # Get the training and validation values
            train_values = list(history_series(history, train_key) or [])
            val_values = list(history_series(history, val_key) or [])

            if train_values:
                # Plot the training curve (x-axis is epoch number, y axis is loss or dive value)
                ax.plot(np.arange(1, len(train_values) + 1), train_values,
                        label = f"{model_name} train", color = colors[model_name], linewidth = 2)
                plotted = True
            if val_values:
                # Plot the validation curve as a dashed line
                ax.plot(np.arange(1, len(val_values) + 1), val_values,
                        label = f"{model_name} validation", color = colors[model_name],
                        linestyle = "--", linewidth = 2)
                plotted = True
        
        # Configure the plot appearance
        ax.set_title(f"{title} curves")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.25)

        if plotted:
            ax.legend(frameon=False) # Shows which line is which
        else:
            ax.text(0.5, 0.5, f"No {title.lower()} history available",
                    ha = "center", va = "center", transform = ax.transAxes) # If no data then show a message
    
    # Adjust spacing between subplots
    plt.tight_layout()
    plt.show()

    # Plot final metrics (bar chart)
    metric_candidates = [ # Each model might have some or all of these metrics
        ("val_loss", "Validation Loss"),
        ("val_dice_coef", "Validation Dice"),
        ("val_bce", "Validation BCE"),
        ("val_hausdorff_distance", "Validation Hausdorff"),
    ]

    # Find which metrics are actually available
    available_metrics = []
    for metric_key, metric_label in metric_candidates:
        for history in histories.values():
            if history_series(history, metric_key) is not None:
                available_metrics.append((metric_key, metric_label))
                break
    
    if available_metrics:
        # Create a bar chart comparing models' final metric values
        labels = list(histories.keys())
        x = np.arange(len(labels))
        width = 0.8 / max(len(available_metrics), 1) # Calculate bar width so multiple metrics don't overlap

        fig, ax = plt.subplots(figzise = (12, 5))

        # For each metric, plot a bar for each model
        for index, (metric_key, metric_label) in enumerate(available_metrics):
            offsets = x + (index - (len (available_metrics) - 1) / 2) * width
            values = []

            # Get the final value of this metric for each model (i.e. the value from the last epoch)
            for model_name in labels:
                history = histories[model_name]
                series = list(history_series(history, metric_key) or [])
                # Take the last value from the series (final metric after training)
                values.append(series[-1] if series else np.nan)
            
            # Plot bars for this metric
            ax.bar(offsets, values, width = width, label = metric_label)
        
        # Configure bar chart appearance
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Final epoch value")
        ax.set_title("Final validation metric comparison")
        ax.grid(axis = "y", alpha = 0.25)
        ax.legend(frameon = False)
        plt.tight_layout()
        plt.show()
    else:
        print("No validation metrics found for a bar chart.")

No saved training histories found. Save each model's history as unet_history.json and swin_history.json to plot learning curves.
